# 생성형 AI 윤리 가이드북 RAG

## 환경 설정

In [ ]:
import json
from dotenv import load_dotenv

load_dotenv()

PDF_PATH = "data/생성형_AI윤리_가이드북_RAG시험용_30쪽.pdf"
QUESTION_PATH = "data/student_13_questions.json"

In [ ]:
f = open(QUESTION_PATH, encoding="utf-8")
data = json.load(f)
f.close()

questions = data["questions"]
print(len(questions))

for q in questions:
    print(q["question"])

## 1단계. 문서 로드

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(PDF_PATH)
pages = loader.load()
print(len(pages))

In [ ]:
for d in pages:
    print(d.metadata["page"] + 1, len(d.page_content.strip()))

In [ ]:
pages = pages[1:]
print(len(pages))
print(pages[0].page_content[:300])

## 2단계. 청킹

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(pages)
print(len(chunks))

## 3단계. 임베딩

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 4단계. 벡터스토어

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)
print(vector_store._collection.count())

In [ ]:
result = vector_store.similarity_search(questions[1]["question"], k=3)
print(result[0].page_content)

## 5단계. 리트리버

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10

chroma_retriever = vector_store.as_retriever(search_kwargs={"k": 10})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.5, 0.5],
)

In [ ]:
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
compressor = CrossEncoderReranker(model=model, top_n=4)

retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=ensemble_retriever,
)

## 6단계. 프롬프트

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
당신은 생성형 AI 윤리 가이드북을 바탕으로 답변하는 도우미입니다.
아래 [문서] 내용만 근거로 질문에 답하세요. 문서에 없는 내용은 추측해서 쓰지 마세요.
질문에 직접 답하는 문장으로 시작하고, 질문과 관계없는 내용은 빼세요.
질문이 요구한 항목(예: 세 가지, 비교, 사례)만 빠짐없이 3~5문장으로 답하세요.

[문서]
{context}

[질문]
{question}

[답변]
""")

## 7단계. LLM

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o", temperature=0)

## 8단계. 출력 파서

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

## 9단계. 체인

In [ ]:
def format_docs(docs):
    text = ""
    for d in docs:
        text += d.page_content + "\n\n"
    return text

In [ ]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | parser
)

In [ ]:
answer = chain.invoke(questions[0]["question"])
print(answer)